# European Fleet Register partitioning by country

## Purpose

This notebook partitions the complete European Fleet Register (EUFR) export
into one country-specific CSV file per value of `Country of Registration`.

The generated files are used as inputs by
`prepare_consolidated_vessel_data.ipynb`, which harmonises EUFR records with
the corresponding national vessel-register data.

## Processing workflow

1. Locate the EFFKG repository root.
2. Load the complete EUFR CSV export as string-valued data.
3. Validate the presence of the country-of-registration field.
4. Exclude rows without a usable country value.
5. Group vessel records by country of registration.
6. Export one semicolon-separated CSV file per country.

## Output

Country-specific files are written to:

`source_data/eufr_data_by_country/eufr_data_<country>.csv`

The notebook prepares source data only. It does not modify Wikibase.

# Setup

In [ ]:
from pathlib import Path
from typing import Optional
import re

import pandas as pd


# =============================================================================
# INITIAL SETUP AND CONFIGURATION
# =============================================================================

def find_repository_root(start: Optional[Path] = None) -> Path:
    """
    Locate the EFFKG repository root.

    The root is identified through the principal directories distributed with
    the project. This implementation is compatible with Python 3.9.
    """
    current = (start or Path.cwd()).resolve()

    required_directories = {
        "code",
        "dataset",
        "schema",
        "source_data",
        "data_model",
        "validation",
    }

    for candidate in [current] + list(current.parents):
        try:
            existing_directories = {
                path.name
                for path in candidate.iterdir()
                if path.is_dir()
            }
        except (PermissionError, OSError):
            continue

        if required_directories.issubset(existing_directories):
            return candidate

    raise RuntimeError(
        "The EFFKG repository root could not be located. "
        "Run this notebook from within a cloned EFFKG repository."
    )


REPOSITORY_ROOT = find_repository_root()

SOURCE_DATA_DIRECTORY = (
    REPOSITORY_ROOT
    / "source_data"
)

INPUT_FILE = (
    SOURCE_DATA_DIRECTORY
    / "eufr_data.csv"
)

OUTPUT_DIRECTORY = (
    SOURCE_DATA_DIRECTORY
    / "eufr_data_by_country"
)

COUNTRY_COLUMN = "Country of Registration"
OUTPUT_PREFIX = "eufr_data"


# =============================================================================
# CONFIGURATION VALIDATION
# =============================================================================

if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        "The complete European Fleet Register export was not found:\n"
        "  {}\n\n"
        "Place the semicolon-separated EUFR CSV in source_data/ "
        "or update INPUT_FILE in the configuration section."
        .format(INPUT_FILE)
    )

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


print("EFFKG European Fleet Register partitioning")
print("-------------------------------------------")
print("Repository root: {}".format(REPOSITORY_ROOT))
print("Input file: {}".format(INPUT_FILE))
print("Country field: {}".format(COUNTRY_COLUMN))
print("Output directory: {}".format(OUTPUT_DIRECTORY))

# Execution

In [ ]:
def safe_filename(value: str) -> str:
    """
    Convert a country value into a safe filename fragment.

    This removes characters that are invalid on common filesystems and
    normalizes whitespace to underscores.
    """
    name = str(value).strip()
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name


def load_eufr_csv(path: Path) -> pd.DataFrame:
    """
    Load the EUFR CSV export as strings.

    Keeping every column as string avoids pandas converting identifiers,
    dates, registration numbers or leading-zero values.
    """
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    df = pd.read_csv(path, sep=";", dtype=str)

    if COUNTRY_COLUMN not in df.columns:
        raise ValueError(
            f"Missing required column '{COUNTRY_COLUMN}'. "
            f"Available columns: {list(df.columns)}"
        )

    return df


def split_by_country(df: pd.DataFrame, output_dir: Path) -> int:
    """
    Write one CSV file per country of registration.

    Rows with missing or empty country values are ignored.
    Returns the number of files generated.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    files_written = 0

    for country, group in df.groupby(COUNTRY_COLUMN, dropna=True):
        country_value = str(country).strip()
        if not country_value:
            continue

        filename = f"{OUTPUT_PREFIX}_{safe_filename(country_value)}.csv"
        filepath = output_dir / filename

        group.to_csv(filepath, sep=";", index=False, encoding="utf-8-sig")

        files_written += 1
        print(f"Saved: {filepath} ({len(group)} rows)")

    return files_written

def main() -> int:
    """
    Load the source EUFR CSV and generate country-level CSV files.
    """
    print(f"Input file: {INPUT_FILE}")
    print(f"Output directory: {OUTPUT_DIRECTORY}")

    df = load_eufr_csv(INPUT_FILE)
    print(f"Rows loaded: {len(df)}")

    files_written = split_by_country(df, OUTPUT_DIRECTORY)
    print(f"Files generated: {files_written}")

    return 0


if __name__ == "__main__":
    main()